<a href="https://colab.research.google.com/github/sk6952217-pixel/Day11_Shivam-Kumar_Vision-Preprocessing/blob/main/Day11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import zipfile # Import the zipfile module

zip_path = "/content/Underwater-Image-Data-set-main.zip" # Define the path to the ZIP file
extract_path = "/content/underwater_dataset" # Define the directory where the files will be extracted

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

In [2]:
# Install 7zip, a command-line archiving tool
!apt-get -qq install p7zip-full

# Define the path to the dataset folder which contains .7z archives
dataset = "/content/underwater_dataset/Underwater-Image-Data-set-main"

output = "/content/images"

!mkdir -p /content/images
!7z x "$dataset/*.7z" -o"$output" -y


7-Zip [64] 16.02 : Copyright (c) 1999-2016 Igor Pavlov : 2016-05-21
p7zip Version 16.02 (locale=en_US.UTF-8,Utf16=on,HugeFiles=on,64 bits,2 CPUs Intel(R) Xeon(R) CPU @ 2.20GHz (406F0),ASM,AES-NI)

Scanning the drive for archives:
  0M Scan /content/underwater_dataset/Underwater-Image-Data-set-main/                                                                     8 files, 151077006 bytes (145 MiB)
  0%    
Extracting archive: /content/underwater_dataset/Underwater-Image-Data-set-main/epoch_00100.7z
--
Path = /content/underwater_dataset/Underwater-Image-Data-set-main/epoch_00100.7z
Type = 7z
Physical Size = 31290555
Headers Size = 8080
Method = LZMA:23
Solid = +
Blocks = 1

  0%      1%      3% 57 - epoch_0100/img 118_gen.png                                    

In [3]:
#Count images
import os

dataset = "/content/images" # Corrected dataset path

#intialize to 0
input_count = 0
target_count = 0
gen_count = 0

#look inside the folder and subfolder
for root, dirs, files in os.walk(dataset):

    for file in files: #check each file one by one

        if "_input.png" in file:  #if file name is "_input.png" input_count increase with 1
            input_count = input_count + 1

        elif "_target.png" in file:  #if file name is "_target.png" target_count increase with 1
            target_count = target_count + 1

        elif "_gen.png" in file:
            gen_count = gen_count + 1

#shows the final count
print("Input images:", input_count)
print("Target images:", target_count)
print("Generated images:", gen_count)
#counts the total
print("Total:", input_count + target_count + gen_count)

Input images: 1078
Target images: 1078
Generated images: 1078
Total: 3234


In [15]:
import os
import shutil
import random

source = "/content/images" # Define the source directory
output = "/content/Dataset_v1" # Define the output directory where the prepared dataset will be stored

random.seed(42) # Set a random seed for reproducibility

# Create folders for train, validation, and test sets, each with 'input' and 'target' subfolders
for split in ["train", "val", "test"]:
    for kind in ["input", "target"]:

        os.makedirs(f"{output}/{split}/{kind}", exist_ok=True)

# Find input-target pairs by scanning the source directory
pairs = {}

# os.walk generates file names in a directory tree
for root, dirs, files in os.walk(source):
    for file in files:

        # Check if the file is an input image
        if file.endswith("_input.png"):
            key = file.replace("_input.png", "") # Extract a common key for pairing (e.g., 'img_1' from 'img_1_input.png')
            pairs.setdefault(key, {})["input"] = os.path.join(root, file)

        # Check if the file is a target image
        elif file.endswith("_target.png"):
            key = file.replace("_target.png", "") # Extract the common key
            # Add the target image path to the corresponding key
            pairs.setdefault(key, {})["target"] = os.path.join(root, file)

# Filter the collected pairs to ensure only complete input-target pairs are kept
pairs = [
    p for p in pairs.values() # Iterate through the values of the 'pairs' dictionary
    if "input" in p and "target" in p # Keep only those dictionaries that contain both 'input' and 'target' keys
]

print("Valid input-target pairs:", len(pairs))


random.shuffle(pairs)

# Define the split ratios for training, validation, and test sets
n = len(pairs)

train_end = int(0.70 * n) # Calculate the end index for the training set 70%
val_end = int(0.85 * n) # Calculate the end index for the validation set (15% + 70% = 85%)

train_pairs = pairs[:train_end]
val_pairs = pairs[train_end:val_end]
test_pairs = pairs[val_end:] # Slice the list to get test pairs (remaining 15%)

print("Train:", len(train_pairs))
print("Validation:", len(val_pairs))
print("Test:", len(test_pairs))

# Define a helper function to copy image pairs to their respective split directories
def copy_pairs(data, split):

    for i, pair in enumerate(data):

        # Copy the input image to the corresponding split's input folder
        shutil.copy(
            pair["input"],
            f"{output}/{split}/input/{i}.png"
        )

        # Copy the target image to the corresponding split's target folder
        shutil.copy(
            pair["target"],
            f"{output}/{split}/target/{i}.png"
        )

# Call the copy_pairs function for each split to organize the dataset
copy_pairs(train_pairs, "train") # Copy training pairs
copy_pairs(val_pairs, "val") # Copy validation pairs
copy_pairs(test_pairs, "test") # Copy test pairs

print("\nDataset V1 created successfully!")

Valid input-target pairs: 312
Train: 218
Validation: 47
Test: 47

Dataset V1 created successfully!


In [16]:
#Resize and Normalize
from PIL import Image
import numpy as np
import os

dataset = "/content/Dataset_v1"

# Define a function to load, resize, and normalize an image
def load_image(path):
    # Open the image from the given path and convert it to RGB format
    img = Image.open(path).convert("RGB")
    # Resize the image to 256x256 pixels. This ensures all images have a consistent size.
    img = img.resize((256, 256))

    img = np.array(img, dtype=np.float32)
    # Normalize the pixel values from the 0-255 range to the 0.0-1.0 range
    img = img / 255.0

    return img


# Define paths to a sample input and target image from the training set
input_path = dataset + "/train/input/0.png"
target_path = dataset + "/train/target/0.png"

# Load and process the sample input image
x = load_image(input_path)
# Load and process the sample target image
y = load_image(target_path)

print("Input shape:", x.shape)
print("Target shape:", y.shape)

print("Input pixel range:", x.min(), "to", x.max())
print("Target pixel range:", y.min(), "to", y.max())

Input shape: (256, 256, 3)
Target shape: (256, 256, 3)
Input pixel range: 0.8509804 to 0.972549
Target pixel range: 0.07450981 to 1.0


In [14]:
#Create Dataset Loader
import torch
from torch.utils.data import Dataset, DataLoader # Tools for creating custom datasets and loading data in batches
from PIL import Image
import os
import numpy as np

# Define a custom dataset class that inherits from torch.utils.data.Dataset
class UnderwaterDataset(Dataset):

    # The constructor for the dataset class
    def __init__(self, folder):
        # Define the paths to the input and target image folders
        self.input_folder = folder + "/input"
        self.target_folder = folder + "/target"

        # Get a sorted list of all file names in the input folder
        self.files = sorted(os.listdir(self.input_folder))

    # Returns the total number of items in the dataset
    def __len__(self):
        return len(self.files)

    # Returns an item (input image and target image pair) at a given index
    def __getitem__(self, index):

        # Get the file name for the current index
        name = self.files[index]

        # Open the input image using its full path and convert it to RGB format
        input_img = Image.open(
            self.input_folder + "/" + name
        ).convert("RGB")

        # Open the target image using its full path and convert it to RGB format
        target_img = Image.open(
            self.target_folder + "/" + name
        ).convert("RGB")

        # Convert the images from PIL Image format to NumPy arrays and normalize pixel values to be between 0 and 1
        input_img = np.array(input_img, dtype=np.float32) / 255.0
        target_img = np.array(target_img, dtype=np.float32) / 255.0

        # Change the image dimension order from H,W,C (Height, Width, Channels) to C,H,W (Channels, Height, Width)
        input_img = torch.tensor(input_img).permute(2, 0, 1)
        target_img = torch.tensor(target_img).permute(2, 0, 1)

        return input_img, target_img


# Create instances of our custom dataset for training, validation, and testing
# The paths point to the respective folders where the prepared images are stored
train_data = UnderwaterDataset(
    "/content/Dataset_v1/train"
)

val_data = UnderwaterDataset(
    "/content/Dataset_v1/val"
)

test_data = UnderwaterDataset(
    "/content/Dataset_v1/test"
)

print("Train:", len(train_data))
print("Validation:", len(val_data))
print("Test:", len(test_data))

# Create DataLoaders for each dataset
train_loader = DataLoader(
    train_data, # The dataset to load from
    batch_size=8, # Number of samples per batch
    shuffle=True # Shuffle the data every epoch for training
)

val_loader = DataLoader(
    val_data, # The validation dataset
    batch_size=8, # Number of samples per batch
    shuffle=False # No need to shuffle validation data
)

test_loader = DataLoader(
    test_data, # The test dataset
    batch_size=8, # Number of samples per batch
    shuffle=False # No need to shuffle test data
)

# Test the data loader by getting one batch from the training loader
x, y = next(iter(train_loader))

print("Input batch shape:", x.shape) # [batch_size, channels, height, width]
print("Target batch shape:", y.shape)

Train: 218
Validation: 47
Test: 47
Input batch shape: torch.Size([8, 3, 256, 256])
Target batch shape: torch.Size([8, 3, 256, 256])


In [13]:
# Create V1 configuration
import json

config = {
    "version": "V1",
    "image_size": [256, 256],
    "image_format": "PNG",
    "channels": 3,
    "normalization": "0-1",
    "train_pairs": 218,
    "validation_pairs": 47,
    "test_pairs": 47,
    "total_pairs": 312,
    "batch_size": 8,
    "augmentation": "None",
    "split": {
        "train": "70%",
        "validation": "15%",
        "test": "15%"
    },
    "random_seed": 42
}

with open(
    "/content/Dataset_v1/preprocessing_config_v1.json",
    "w"
) as f:
    json.dump(config, f, indent=4)

print("Preprocessing configuration saved!")

Preprocessing configuration saved!


In [17]:
!zip -r dataset.zip /content/Dataset_v1/

  adding: content/Dataset_v1/ (stored 0%)
  adding: content/Dataset_v1/test/ (stored 0%)
  adding: content/Dataset_v1/test/input/ (stored 0%)
  adding: content/Dataset_v1/test/input/32.png (deflated 4%)
  adding: content/Dataset_v1/test/input/20.png (deflated 2%)
  adding: content/Dataset_v1/test/input/26.png (deflated 15%)
  adding: content/Dataset_v1/test/input/4.png (deflated 2%)
  adding: content/Dataset_v1/test/input/37.png (deflated 15%)
  adding: content/Dataset_v1/test/input/24.png (deflated 3%)
  adding: content/Dataset_v1/test/input/27.png (deflated 2%)
  adding: content/Dataset_v1/test/input/43.png (deflated 4%)
  adding: content/Dataset_v1/test/input/14.png (deflated 13%)
  adding: content/Dataset_v1/test/input/16.png (deflated 2%)
  adding: content/Dataset_v1/test/input/30.png (deflated 17%)
  adding: content/Dataset_v1/test/input/9.png (deflated 15%)
  adding: content/Dataset_v1/test/input/35.png (deflated 6%)
  adding: content/Dataset_v1/test/input/6.png (deflated 2%)
  